<center>
<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/M5_Final/images/SN_web_lightmode.png" width="300">
</center>


<h1>Analysis of Global COVID-19 Pandemic Data</h1>

Estimated time needed: **90** minutes



## Overview:

In this final project, you will apply your knowledge of data analysis by completing a series of hands-on tasks in this lab notebook. The lab consists of 10 tasks, where you will write and execute code to demonstrate your skills.

While working through the lab, remember to save all your code and outputs, then download the Jupyter Notebook, as you will need to submit the completed notebook for the **Final Project: Submission and Evaluation**.

If you need to refresh your memories about specific coding details, you may refer to previous hands-on labs for code examples.


In [26]:
# This lab requires 'httr' and 'rvest'packages, which are already pre-loaded into this lab environment.
# However, if you are working on your local RStudio, please uncomment the below codes and install the packages.

# install.packages("httr")
# install.packages("rvest")

In [27]:
library(httr)
library(rvest)

Note: if you can import above libraries, please use install.packages() to install them first.


## TASK 1: Get a `COVID-19 pandemic` Wiki page using HTTP request


First, let's write a function to use HTTP request to get a public COVID-19 Wiki page.

Before you write the function, you can open this public page from this 

URL https://en.wikipedia.org/w/index.php?title=Template:COVID-19_testing_by_country using a web browser.

The goal of task 1 is to get the html page using HTTP request (`httr` library)


In [28]:

get_wiki_covid19_page <- function() {
    
  # Our target COVID-19 wiki page URL is: https://en.wikipedia.org/w/index.php?title=Template:COVID-19_testing_by_country  
  # Which has two parts: 
    # 1) base URL `https://en.wikipedia.org/w/index.php  
    # 2) URL parameter: `title=Template:COVID-19_testing_by_country`, seperated by question mark ?
    
  # Wiki page base
  wiki_base_url <- "https://en.wikipedia.org/w/index.php"
  # You will need to create a List which has an element called `title` to specify which page you want to get from Wiki
  # in our case, it will be `Template:COVID-19_testing_by_country`
  wiki_query_params <- list(title = "Template:COVID-19_testing_by_country")
 
  # - Use the `GET` function in httr library with a `url` argument and a `query` arugment to get a HTTP response
  response <- GET(url = wiki_base_url, query = wiki_query_params)
    
  # Use the `return` function to return the response
  return(response)
}

Call the `get_wiki_covid19_page` function to get a http response with the target html page


In [29]:
# Call the get_wiki_covid19_page function and print the response
get_wiki_covid19_page()

Response [https://en.wikipedia.org/w/index.php?title=Template%3ACOVID-19_testing_by_country]
  Date: 2026-01-29 17:30
  Status: 200
  Content-Type: text/html; charset=UTF-8
  Size: 461 kB
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-fea...
<head>
<meta charset="UTF-8">
<title>Template:COVID-19 testing by country - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-heade...
RLSTATE={"ext.globalCssJs.user.styles":"ready","site.styles":"ready","user.st...
<script>(RLQ=window.RLQ||[]).push(function(){mw.loader.impl(function(){return...
}];});});</script>
<link rel="stylesheet" href="/w/load.php?lang=en&amp;modules=ext.cite.styles%...
...

## TASK 2: Extract COVID-19 testing data table from the wiki HTML page


On the COVID-19 testing wiki page, you should see a data table `<table>` node contains COVID-19 testing data by country on the page:

<a href="https://cognitiveclass.ai/">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/M5_Final/images/covid-19-by-country.png" width="400" align="center">
</a>

Note the numbers you actually see on your page may be different from above because it is still an on-going pandemic when creating this notebook.

The goal of task 2 is to extract above data table and convert it into a data frame


Now use the `read_html` function in rvest library to get the root html node from response


In [30]:
# Get the root html node from the http response in task 1 
wiki_node <- read_html(get_wiki_covid19_page())
wiki_node

{html_document}
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available" lang="en" dir="ltr">
[1] <head>\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8 ...
[2] <body class="skin--responsive skin-vector skin-vector-search-vue mediawik ...

Get the tables in the HTML root node using `html_nodes` function.


In [31]:
# Get the table node from the root html node
table_node <- html_nodes(wiki_node, "table")
table_node

{xml_nodeset (3)}
[1] <table class="wikitable plainrowheaders sortable collapsible autocollapse ...
[2] <table class="plainlinks ombox mbox-small ombox-notice" role="presentatio ...
[3] <table class="wikitable mw-templatedata-doc-params">\n<caption><p class=" ...

Read the specific table from the multiple tables in the `table_node` using the `html_table` function and convert it into dataframe using `as.data.frame`

_Hint:- Please read the `table_node` with index 2(ex:- table_node[2])._


In [32]:
# Read the table node and convert it into a data frame, and print the data frame for review
data_frame <- as.data.frame(html_table(table_node[2]))
head(data_frame)

,X1,X2
,<lgl>,<chr>
1,NA,This template uses TemplateStyles: Template:COVID-19 testing by country/styles.css


## TASK 3: Pre-process and export the extracted data frame

The goal of task 3 is to pre-process the extracted data frame from the previous step, and export it as a csv file


Let's get a summary of the data frame


In [33]:
# Print the summary of the data frame
summary(data_frame)

    X1               X2           
 Mode:logical   Length:1          
 NA's:1         Class :character  
                Mode  :character  

As you can see from the summary, the columns names are little bit different to understand and some column data types are not correct. For example, the `Tested` column shows as `character`. 

As such, the data frame read from HTML table will need some pre-processing such as removing irrelvant columns, renaming columns, and convert columns into proper data types.


We have prepared a pre-processing function for you to conver the data frame but you can also try to write one by yourself


In [34]:
preprocess_covid_data_frame <- function(data_frame) {
    
    shape <- dim(data_frame)

    # Remove the World row
    data_frame<-data_frame[!(data_frame$`Country.or.region`=="World"),]
    # Remove the last row
    data_frame <- data_frame[1:172, ]
    
    # We dont need the Units and Ref columns, so can be removed
    data_frame["Ref."] <- NULL
    data_frame["Units.b."] <- NULL
    
    # Renaming the columns
    names(data_frame) <- c("country", "date", "tested", "confirmed", "confirmed.tested.ratio", "tested.population.ratio", "confirmed.population.ratio")
    
    # Convert column data types
    data_frame$country <- as.factor(data_frame$country)
    data_frame$date <- as.factor(data_frame$date)
    data_frame$tested <- as.numeric(gsub(",","",data_frame$tested))
    data_frame$confirmed <- as.numeric(gsub(",","",data_frame$confirmed))
    data_frame$'confirmed.tested.ratio' <- as.numeric(gsub(",","",data_frame$`confirmed.tested.ratio`))
    data_frame$'tested.population.ratio' <- as.numeric(gsub(",","",data_frame$`tested.population.ratio`))
    data_frame$'confirmed.population.ratio' <- as.numeric(gsub(",","",data_frame$`confirmed.population.ratio`))
    
    return(data_frame)
}


Call the `preprocess_covid_data_frame` function


In [35]:
# call `preprocess_covid_data_frame` function and assign it to a new data frame
# proper_data_frame <- preprocess_covid_data_frame(data_frame)
# head(proper_data_frame)

Get the summary of the processed data frame again


In [36]:
# Print the summary of the processed data frame again
# summary(proper_data_frame)

After pre-processing, you can see the columns and columns names are simplified, and columns types are converted into correct types.


The data frame has following columns:

- **country** - The name of the country
- **date** - Reported date
- **tested** - Total tested cases by the reported date
- **confirmed** - Total confirmed cases by the reported date
- **confirmed.tested.ratio** - The ratio of confirmed cases to the tested cases
- **tested.population.ratio** - The ratio of tested cases to the population of the country
- **confirmed.population.ratio** - The ratio of confirmed cases to the population of the country


OK, we can call `write.csv()` function to save the csv file into a file. 


In [37]:
# Export the data frame to a csv file
# write.csv(proper_data_frame, "covid.csv", row.names = FALSE)

Note for IBM Waston Studio, there is no traditional "hard disk" associated with a R workspace.

Even if you call `write.csv()` method to save the data frame as a csv file, it won't be shown in IBM Cloud Object Storage asset UI automatically.

However, you may still check if the `covid.csv` exists using following code snippet:


In [38]:
# Get working directory
wd <- getwd()
# Get exported 
file_path <- paste(wd, sep="", "/covid.csv")
# File path
print(file_path)
file.exists(file_path)

[1] "/resources/RP0101EN/labs/M5/covid.csv"


[1] TRUE

**Optional Step**: If you have difficulties finishing above webscraping tasks, you may still continue with next tasks by downloading a provided csv file from here:


In [39]:
## Download a sample csv file
covid_csv_file <- download.file("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/dataset/covid.csv", destfile="covid.csv")
covid_data_frame_csv <- read.csv("covid.csv", header=TRUE, sep=",")
head(covid_data_frame_csv)
summary(covid_data_frame_csv)

,country,date,tested,confirmed,confirmed.tested.ratio,tested.population.ratio,confirmed.population.ratio
,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
1,Afghanistan,17 Dec 2020,154767,49621,32.1,0.40,0.130
2,Albania,18 Feb 2021,428654,96838,22.6,15.00,3.400
3,Algeria,2 Nov 2020,230553,58574,25.4,0.53,0.130
4,Andorra,8 Mar 2021,159725,11066,6.9,206.00,14.300
5,Angola,12 Mar 2021,399228,20981,5.3,1.30,0.067
6,Antigua and Barbuda,6 Mar 2021,15268,832,5.4,15.90,0.860


   country              date               tested            confirmed       
 Length:172         Length:172         Min.   :     3880   Min.   :       0  
 Class :character   Class :character   1st Qu.:   229662   1st Qu.:   10924  
 Mode  :character   Mode  :character   Median :  1035758   Median :   86514  
                                       Mean   :  9875472   Mean   :  682058  
                                       3rd Qu.:  5391959   3rd Qu.:  339156  
                                       Max.   :362125287   Max.   :29403102  
 confirmed.tested.ratio tested.population.ratio confirmed.population.ratio
 Min.   : 0.00          Min.   :  0.0065        Min.   : 0.0000           
 1st Qu.: 3.40          1st Qu.:  3.8250        1st Qu.: 0.1375           
 Median : 7.80          Median : 16.1500        Median : 1.2500           
 Mean   :10.15          Mean   : 41.8488        Mean   : 2.5078           
 3rd Qu.:14.62          3rd Qu.: 48.1500        3rd Qu.: 4.4250           
 Max

## TASK 4: Get a subset of the extracted data frame

The goal of task 4 is to get the 5th to 10th rows from the data frame with only `country` and `confirmed` columns selected


In [40]:
# Read covid_data_frame_csv from the csv file
covid_data <- read.csv("covid.csv")
# Get the 5th to 10th rows, with two "country" "confirmed" columns
covid_data[5:10,c('country','confirmed')]

,country,confirmed
,<chr>,<int>
5,Angola,20981
6,Antigua and Barbuda,832
7,Argentina,2195722
8,Armenia,177104
9,Australia,29130
10,Austria,488007


## TASK 5: Calculate worldwide COVID testing positive ratio

The goal of task 5 is to get the total confirmed and tested cases worldwide, and try to figure the overall positive ratio using `confirmed cases / tested cases`


In [41]:
# Get the total confirmed cases worldwide
tot_confirmed <- sum(covid_data[,'confirmed'])
tot_confirmed
# Get the total tested cases worldwide
tot_tested <- sum(covid_data[,'tested'])
tot_tested
# Get the positive ratio (confirmed / tested)
positive_ratio <- tot_confirmed/tot_tested
round(positive_ratio,2)

[1] 117313932

[1] 1698581244

[1] 0.07

## TASK 6: Get a country list which reported their testing data 

The goal of task 6 is to get a catalog or sorted list of countries who have reported their COVID-19 testing data


In [42]:
# Get the `country` column
covid_data[,'country']
# Check its class (should be Factor)
class(covid_data$country)
# Convert the country column into character so that you can easily sort them
covid_data$country <- as.character(covid_data$country)
class(covid_data$country)
# Sort the countries AtoZ
sort(covid_data$country)
# Sort the countries ZtoA
ztoa_country <- sort(covid_data$country, decreasing=TRUE)
# Print the sorted ZtoA list
print(ztoa_country)

[1] "Afghanistan"            "Albania"                "Algeria"               
  [4] "Andorra"                "Angola"                 "Antigua and Barbuda"   
  [7] "Argentina"              "Armenia"                "Australia"             
 [10] "Austria"                "Azerbaijan"             "Bahamas"               
 [13] "Bahrain"                "Bangladesh"             "Barbados"              
 [16] "Belarus"                "Belgium"                "Belize"                
 [19] "Benin"                  "Bhutan"                 "Bolivia"               
 [22] "Bosnia and Herzegovina" "Botswana"               "Brazil"                
 [25] "Brunei"                 "Bulgaria"               "Burkina Faso"          
 [28] "Burundi"                "Cambodia"               "Cameroon"              
 [31] "Canada"                 "Chad"                   "Chile"                 
 [34] "China[c]"               "Colombia"               "Costa Rica"            
 [37] "Croatia"                "Cuba"                   "Cyprus[d]"             
 [40] "Czechia"                "Denmark[e]"             "Djibouti"              
 [43] "Dominica"               "Dominican Republic"     "DR Congo"              
 [46] "Ecuador"                "Egypt"                  "El Salvador"           
 [49] "Equatorial Guinea"      "Estonia"                "Eswatini"              
 [52] "Ethiopia"               "Faroe Islands"          "Fiji"                  
 [55] "Finland"                "France[f][g]"           "Gabon"                 
 [58] "Gambia"                 "Georgia[h]"             "Germany"               
 [61] "Ghana"                  "Greece"                 "Greenland"             
 [64] "Grenada"                "Guatemala"              "Guinea"                
 [67] "Guinea-Bissau"          "Guyana"                 "Haiti"                 
 [70] "Honduras"               "Hungary"                "Iceland"               
 [73] "India"                  "Indonesia"              "Iran"                  
 [76] "Iraq"                   "Ireland"                "Israel"                
 [79] "Italy"                  "Ivory Coast"            "Jamaica"               
 [82] "Japan"                  "Jordan"                 "Kazakhstan"            
 [85] "Kenya"                  "Kosovo"                 "Kuwait"                
 [88] "Kyrgyzstan"             "Laos"                   "Latvia"                
 [91] "Lebanon"                "Lesotho"                "Liberia"               
 [94] "Libya"                  "Lithuania"              "Luxembourg[i]"         
 [97] "Madagascar"             "Malawi"                 "Malaysia"              
[100] "Maldives"               "Mali"                   "Malta"                 
[103] "Mauritania"             "Mauritius"              "Mexico"                
[106] "Moldova[j]"             "Mongolia"               "Montenegro"            
[109] "Morocco"                "Mozambique"             "Myanmar"               
[112] "Namibia"                "Nepal"                  "Netherlands"           
[115] "New Caledonia"          "New Zealand"            "Niger"                 
[118] "Nigeria"                "North Korea"            "North Macedonia"       
[121] "Northern Cyprus[k]"     "Norway"                 "Oman"                  
[124] "Pakistan"               "Palestine"              "Panama"                
[127] "Papua New Guinea"       "Paraguay"               "Peru"                  
[130] "Philippines"            "Poland"                 "Portugal"              
[133] "Qatar"                  "Romania"                "Russia"                
[136] "Rwanda"                 "Saint Kitts and Nevis"  "Saint Lucia"           
[139] "Saint Vincent"          "San Marino"             "Saudi Arabia"          
[142] "Senegal"                "Serbia"                 "Singapore"             
[145] "Slovakia"               "Slovenia"               "South Africa"          
[148] "South Korea"            "S

[1] "character"

[1] "character"

[1] "Afghanistan"            "Albania"                "Algeria"               
  [4] "Andorra"                "Angola"                 "Antigua and Barbuda"   
  [7] "Argentina"              "Armenia"                "Australia"             
 [10] "Austria"                "Azerbaijan"             "Bahamas"               
 [13] "Bahrain"                "Bangladesh"             "Barbados"              
 [16] "Belarus"                "Belgium"                "Belize"                
 [19] "Benin"                  "Bhutan"                 "Bolivia"               
 [22] "Bosnia and Herzegovina" "Botswana"               "Brazil"                
 [25] "Brunei"                 "Bulgaria"               "Burkina Faso"          
 [28] "Burundi"                "Cambodia"               "Cameroon"              
 [31] "Canada"                 "Chad"                   "Chile"                 
 [34] "China[c]"               "Colombia"               "Costa Rica"            
 [37] "Croatia"                "Cuba"                   "Cyprus[d]"             
 [40] "Czechia"                "Denmark[e]"             "Djibouti"              
 [43] "Dominica"               "Dominican Republic"     "DR Congo"              
 [46] "Ecuador"                "Egypt"                  "El Salvador"           
 [49] "Equatorial Guinea"      "Estonia"                "Eswatini"              
 [52] "Ethiopia"               "Faroe Islands"          "Fiji"                  
 [55] "Finland"                "France[f][g]"           "Gabon"                 
 [58] "Gambia"                 "Georgia[h]"             "Germany"               
 [61] "Ghana"                  "Greece"                 "Greenland"             
 [64] "Grenada"                "Guatemala"              "Guinea"                
 [67] "Guinea-Bissau"          "Guyana"                 "Haiti"                 
 [70] "Honduras"               "Hungary"                "Iceland"               
 [73] "India"                  "Indonesia"              "Iran"                  
 [76] "Iraq"                   "Ireland"                "Israel"                
 [79] "Italy"                  "Ivory Coast"            "Jamaica"               
 [82] "Japan"                  "Jordan"                 "Kazakhstan"            
 [85] "Kenya"                  "Kosovo"                 "Kuwait"                
 [88] "Kyrgyzstan"             "Laos"                   "Latvia"                
 [91] "Lebanon"                "Lesotho"                "Liberia"               
 [94] "Libya"                  "Lithuania"              "Luxembourg[i]"         
 [97] "Madagascar"             "Malawi"                 "Malaysia"              
[100] "Maldives"               "Mali"                   "Malta"                 
[103] "Mauritania"             "Mauritius"              "Mexico"                
[106] "Moldova[j]"             "Mongolia"               "Montenegro"            
[109] "Morocco"                "Mozambique"             "Myanmar"               
[112] "Namibia"                "Nepal"                  "Netherlands"           
[115] "New Caledonia"          "New Zealand"            "Niger"                 
[118] "Nigeria"                "North Korea"            "North Macedonia"       
[121] "Northern Cyprus[k]"     "Norway"                 "Oman"                  
[124] "Pakistan"               "Palestine"              "Panama"                
[127] "Papua New Guinea"       "Paraguay"               "Peru"                  
[130] "Philippines"            "Poland"                 "Portugal"              
[133] "Qatar"                  "Romania"                "Russia"                
[136] "Rwanda"                 "Saint Kitts and Nevis"  "Saint Lucia"           
[139] "Saint Vincent"          "San Marino"             "Saudi Arabia"          
[142] "Senegal"                "Serbia"                 "Singapore"             
[145] "Slovakia"               "Slovenia"               "South Africa"          
[148] "South Korea"            "S

  [1] "Zimbabwe"               "Zambia"                 "Vietnam"               
  [4] "Venezuela"              "Uzbekistan"             "Uruguay"               
  [7] "United States"          "United Kingdom"         "United Arab Emirates"  
 [10] "Ukraine"                "Uganda"                 "Turkey"                
 [13] "Tunisia"                "Trinidad and Tobago"    "Togo"                  
 [16] "Thailand"               "Tanzania"               "Taiwan[m]"             
 [19] "Switzerland[l]"         "Sweden"                 "Sudan"                 
 [22] "Sri Lanka"              "Spain"                  "South Sudan"           
 [25] "South Korea"            "South Africa"           "Slovenia"              
 [28] "Slovakia"               "Singapore"              "Serbia"                
 [31] "Senegal"                "Saudi Arabia"           "San Marino"            
 [34] "Saint Vincent"          "Saint Lucia"            "Saint Kitts and Nevis" 
 [37] "Rwanda"              

## TASK 7: Identify countries names with a specific pattern

The goal of task 7 is using a regular expression to find any countires start with `United`


In [43]:
# Use a regular expression `United.+` to find matches
country_matches <- regexpr('United.+', covid_data$country)

# Print the matched country names
regmatches(covid_data$country, country_matches)

[1] "United Arab Emirates" "United Kingdom"       "United States"

## TASK 8: Pick two countries you are interested, and then review their testing data

The goal of task 8 is to compare the COVID-19 test data between two countires, you will need to select two rows from the dataframe, and select `country`, `confirmed`, `confirmed-population-ratio` columns


In [44]:
# Select a subset (should be only one row) of data frame based on a selected country name and columns
india <- covid_data[covid_data$country=='India',c('country','tested','confirmed','confirmed.population.ratio')]
india
# Select a subset (should be only one row) of data frame based on a selected country name and columns
usa <- covid_data[covid_data$country=='United States',c('country','tested','confirmed','confirmed.population.ratio')]
usa

,country,tested,confirmed,confirmed.population.ratio
,<chr>,<dbl>,<int>,<dbl>
73,India,226703641,11359048,0.82


,country,tested,confirmed,confirmed.population.ratio
,<chr>,<dbl>,<int>,<dbl>
166,United States,362125287,29403102,8.9


## TASK 9: Compare which one of the selected countries has a larger ratio of confirmed cases to population

The goal of task 9 is to find out which country you have selected before has larger ratio of confirmed cases to population, which may indicate that country has higher COVID-19 infection risk


In [45]:
if (usa$`confirmed.population.ratio` > india$`confirmed.population.ratio`) {
   print('According to data USA higher covid-19 infection risk as compared to India')
} else {
   print('According to data India has higher covid-19 infection risk as compared to USA')
}

[1] "According to data USA higher covid-19 infection risk as compared to India"


## TASK 10: Find countries with confirmed to population ratio rate less than a threshold

The goal of task 10 is to find out which countries have the confirmed to population ratio less than 1%, it may indicate the risk of those countries are relatively low


In [46]:
# Get a subset of any countries with `confirmed.population.ratio` less than the threshold
new_df <- covid_data[(covid_data$`confirmed.population.ratio` <1), ]
head(new_df)

,country,date,tested,confirmed,confirmed.tested.ratio,tested.population.ratio,confirmed.population.ratio
,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
1,Afghanistan,17 Dec 2020,154767,49621,32.1,0.40,0.130
3,Algeria,2 Nov 2020,230553,58574,25.4,0.53,0.130
5,Angola,12 Mar 2021,399228,20981,5.3,1.30,0.067
6,Antigua and Barbuda,6 Mar 2021,15268,832,5.4,15.90,0.860
9,Australia,15 Mar 2021,14933604,29130,0.2,59.50,0.120
14,Bangladesh,5 Mar 2021,4119031,549184,13.3,2.50,0.330
